# R7-E2 - Robustez con validación cruzada de 5 folds

**Objetivo.** Comprobar si las dos comparaciones de R7 (ConvNeXt-Tiny-A frente a DenseNet-A, y B0 frente a B2) se mantienen cuando cambia la partición de validation, aplicando de forma literal las reglas de lectura escritas de antemano en `results/classification/r7_optimizacion/robustez_e2/protocolo_robustez_e2.json` (commit `6de5349`). Este notebook no cambia ninguna regla.

**Diseño.** Tres configuraciones por cinco folds estratificados (15 corridas, semilla 42). ConvNeXt-A y B0 son la misma configuración y se comparten entre C1 y C2. La métrica del veredicto es el AP; la ROC-AUC se reporta de forma descriptiva.

| Configuración | Arquitectura | Exponente α | Comparaciones |
|---|---|---|---|
| `convnext_a_alpha1` (B0) | ConvNeXt-Tiny, A | 1.0 | C1 y C2 |
| `convnext_a_alpha05` (B2) | ConvNeXt-Tiny, A | 0.5 | C2 |
| `densenet_a_alpha1` | DenseNet-121, A | 1.0 | C1 |

E2 es estratificada y no por paciente: puede sobreestimar y sirve solo como chequeo de robustez, no como estimación final. No se usa test.

## 1. Configuración

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError("No se pudo localizar la raíz del proyecto.")


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification import robustez_e2_analysis as e2  # toda la lógica del análisis está en este módulo

E2_DIR = PROJECT_ROOT / e2.E2_REL
ANALYSIS_DIR = E2_DIR / e2.ANALYSIS_NAME
OUTPUTS = {key: ANALYSIS_DIR / name for key, name in e2.OUTPUT_FILES.items()}

PROTOCOL, PROTOCOL_PATH = e2.load_protocol(PROJECT_ROOT)
RUNS = PROTOCOL["comandos_para_falcon"]["corridas"]
COMPARISONS = e2.parse_comparisons(PROTOCOL)
CONFIGS = list(PROTOCOL["configuraciones"])
N_FOLDS = e2.N_FOLDS

LABELS = {"convnext_a_alpha1": "ConvNeXt-A / B0 (α = 1.0)", "convnext_a_alpha05": "ConvNeXt-A / B2 (α = 0.5)", "densenet_a_alpha1": "DenseNet-A (α = 1.0)"}
COLORS = {"convnext_a_alpha1": "#0072B2", "convnext_a_alpha05": "#009E73", "densenet_a_alpha1": "#D55E00"}
CRITERION_LABELS = {"auc_max": "máx ROC-AUC (vigente)", "ap_max": "máx AP", "loss_min": "mín pérdida ponderada"}

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def require(condition, message):
    if not condition:
        raise AssertionError(message)


print(f"Protocolo: {PROTOCOL_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Estado declarado: {PROTOCOL['estado']}")
print(f"Corridas: {len(RUNS)} | comparaciones: {', '.join(c['id'] for c in COMPARISONS)}")

## 2. Protocolo y comandos de entrenamiento

Las 15 corridas y su comando están escritos en el protocolo; aquí solo se listan y se indica cuáles ya tienen resultados. Con `RUN_TRAINING = False` no se inicia ningún entrenamiento. Con `True` (solo en Falcon) se lanzan las que falten, sin sobrescribir carpetas con archivos.

In [ ]:
RUN_TRAINING = False  # True para lanzar desde este notebook las corridas que falten (Falcon); False solo las lista

done = [run["id"] for run in RUNS if (PROJECT_ROOT / run["salida"] / "validation_metrics.json").exists()]
print(f"Corridas con resultados: {len(done)} de {len(RUNS)}")
print("Ejemplo de comando:\n" + RUNS[0]["comando"])

if not RUN_TRAINING:
    print("\nRUN_TRAINING = False: no se inicia ningún entrenamiento.")
else:
    for run in RUNS:
        output_dir = PROJECT_ROOT / run["salida"]
        print("=" * 80)
        print(f"{run['id']} -> {run['salida']}/")

        if output_dir.exists() and any(output_dir.iterdir()):
            print("Omitida: la carpeta ya contiene archivos y no se sobrescribe. Si una corrida quedó incompleta, revísela o muévala manualmente.")
            continue

        command = [sys.executable, *shlex.split(run["comando"])[1:]]
        process = subprocess.Popen(
            command, cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end="")
        except KeyboardInterrupt:
            process.terminate()  # al interrumpir la celda no debe quedar un entrenamiento huérfano
            raise
        if process.wait() != 0:
            raise RuntimeError(f"La corrida {run['id']} terminó con error (código {process.returncode}).")

## 3. Carga y verificación de las 15 corridas

Esta sección solo lee archivos. Para cada corrida comprueba que la configuración registrada coincide con el comando del protocolo (arquitectura, semilla, exponente α, hiperparámetros, criterio de checkpoint, early stopping), que el manifiesto del fold no incluye test y que su validation es el fold de `cv_folds_e2.csv`, que los pesos de clase corresponden a las etiquetas de train del fold, y que AP y ROC-AUC de `validation_metrics.json` se reproducen desde las probabilidades guardadas y desde el historial. También comprueba que el protocolo es idéntico al del commit de referencia. Si alguna comprobación falla, no se escribe ningún resultado.

Las tablas se escriben en `robustez_e2/analisis/` mediante `src/classification/robustez_e2_analysis.py`; si ya existen, solo se leen.

In [ ]:
missing_runs = [run["salida"] for run in RUNS if not (PROJECT_ROOT / run["salida"] / "validation_metrics.json").exists()]
require(not missing_runs, "Faltan resultados E2 (git pull o completar las corridas):\n" + "\n".join(missing_runs))

existing_outputs = [path for path in OUTPUTS.values() if path.exists()]
if not existing_outputs:
    exit_code, _ = e2.run_analysis(PROJECT_ROOT, verbose=False)
    require(exit_code == 0, "El análisis no se escribió: hay comprobaciones fallidas (ejecute `python -m src.classification.robustez_e2_analysis`).")
else:
    require(len(existing_outputs) == len(OUTPUTS), "analisis/ está incompleta; revise la carpeta antes de continuar.")

verification = pd.read_csv(OUTPUTS["verification"])
summary_checks = verification.groupby("comprobacion", sort=False)["ok"].agg(corridas="size", todas_ok="all")
print(f"Comprobaciones: {len(verification)} | fallidas: {int((~verification['ok']).sum())}")
display(summary_checks)

## 4. AP y ROC-AUC por configuración y fold

AP y ROC-AUC del checkpoint de cada corrida (máximo `val_roc_auc` del fold). No dependen del umbral. Cada fold tiene 200 imágenes y 21 o 22 positivas operacionales, de modo que el AP por fold es ruidoso; la media no es una estimación final del desempeño.

In [ ]:
metrics = pd.read_csv(OUTPUTS["metrics"])

for column, title in (("ap", "AP"), ("roc_auc", "ROC-AUC")):
    table = metrics.pivot(index="configuracion", columns="fold", values=column).loc[CONFIGS]
    table["media"] = table.mean(axis=1)
    table.index = [LABELS[c] for c in table.index]
    print(title)
    display(table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for config in CONFIGS:
    rows = metrics[metrics["configuracion"] == config].sort_values("fold")
    ax.plot(rows["fold"], rows["ap"], color=COLORS[config], marker="o", label=LABELS[config])
ax.set_xticks(range(N_FOLDS))
ax.set_xlabel("Fold (validation)")
ax.set_ylabel("Average Precision")
ax.set_title("Figura 1. AP por fold y configuración (E2)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "fig1_ap_por_fold.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Comparaciones C1 y C2

El delta por fold es la primera configuración menos la segunda, con las mismas imágenes de validation: C1 = ConvNeXt-A - DenseNet-A; C2 = B2 - B0. Se reportan media, mínimo, máximo y conteo de signos; un delta exactamente 0 no cuenta en ningún signo. Con cinco folds no se reporta p-valor ni intervalo de confianza.

In [ ]:
delta_summary = pd.read_csv(OUTPUTS["delta_summary"])
display(
    delta_summary.rename(columns={"folds_positivos": "folds +", "folds_negativos": "folds -", "folds_cero": "folds 0"})
    [["comparacion", "metrica", "media", "minimo", "maximo", "folds +", "folds -", "folds 0"]]
    .round(4)
    .set_index(["comparacion", "metrica"])
)

In [ ]:
delta_by_fold = pd.read_csv(OUTPUTS["delta_by_fold"])
ap_deltas = delta_by_fold[delta_by_fold["metrica"] == "AP"]

fig, axes = plt.subplots(1, len(COMPARISONS), figsize=(11, 4.2), sharey=True)
for ax, comparison in zip(axes, COMPARISONS):
    rows = ap_deltas[ap_deltas["comparacion"] == comparison["id"]].sort_values("fold")
    ax.bar(rows["fold"], rows["delta"], color=COLORS[comparison["first"]], alpha=0.85)
    ax.axhline(0, color="black", lw=1)
    ax.axhline(rows["delta"].mean(), color="black", ls="--", lw=1, label=f"media = {rows['delta'].mean():+.3f}")
    ax.set_xticks(range(N_FOLDS))
    ax.set_xlabel("Fold")
    ax.set_title(f"{comparison['id']}: {LABELS[comparison['first']]}\nmenos {LABELS[comparison['second']]}", fontsize=9)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Δ AP (primera − segunda)")
fig.suptitle("Figura 2. Diferencia de AP por fold", y=1.02)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "fig2_delta_ap_por_fold.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Aplicación literal de las reglas del protocolo

Se aplican la regla de consistencia (mismo signo estricto en al menos 4 de 5 folds), la regla de no declarar «mejor» sin consistencia y `regla_robustez_de_b0`. El protocolo no define una regla específica para C2, de modo que C2 se lee con las dos primeras; no se añade ninguna regla después de ver los resultados. El protocolo fija el texto del veredicto solo para `regla_robustez_de_b0`; las frases de C1 y C2 son formulaciones derivadas de las reglas literales.

In [ ]:
rules = json.loads(OUTPUTS["rules"].read_text(encoding="utf-8"))
print(f"Protocolo idéntico al commit {rules['protocolo']['commit_de_referencia']}: {rules['protocolo']['idéntico_al_commit']}")
print(f"Métrica del veredicto: {rules['metrica_del_veredicto']}\n")

for cid, block in rules["comparaciones"].items():
    print(f"[{cid}] consistente = {block['consistente']} | {block['frase_del_veredicto']}")

print(f"\n[regla_robustez_de_b0] se cumple = {rules['regla_robustez_de_b0']['se_cumple']}")
print(rules["regla_robustez_de_b0"]["frase_del_veredicto"])
print(f"\n[regla_c2] existe en el protocolo = {rules['regla_c2']['existe_en_el_protocolo']}")

## 7. Sensibilidad del criterio de checkpoint por fold

El criterio oficial se mantiene (máx ROC-AUC). Usando los `training_history.json` de E2 y `checkpoint_sensitivity.py`, se calcula qué época habría elegido cada criterio (máx ROC-AUC, máx AP, mín pérdida ponderada) y el AP en esa época, para repetir C1 y C2. Es descriptivo y no modifica el veredicto. Elegir por AP del propio fold y reportar ese AP es optimista.

In [ ]:
sens_by_fold = pd.read_csv(OUTPUTS["sens_by_fold"])
sens_summary = pd.read_csv(OUTPUTS["sens_summary"])

epochs = (
    sens_by_fold.sort_values(["configuracion", "criterio", "fold"])
    .groupby(["configuracion", "criterio"])["epoca_elegida"].agg(lambda s: "/".join(map(str, s)))
    .unstack("criterio").loc[CONFIGS, list(CRITERION_LABELS)]
)
epochs.index = [LABELS[c] for c in epochs.index]
print("Época elegida por criterio (folds 0 a 4)")
display(epochs)

display(
    sens_summary.assign(criterio=lambda d: d["criterio"].map(CRITERION_LABELS))
    .rename(columns={"folds_positivos": "folds +", "folds_negativos": "folds -", "folds_con_signo_distinto_a_auc_max": "folds con signo distinto"})
    [["comparacion", "criterio", "media", "folds +", "folds -", "consistente_4_de_5", "folds con signo distinto",
      "signo_de_la_media_distinto_a_auc_max", "veredicto_de_consistencia_distinto_a_auc_max"]]
    .round(4)
    .set_index(["comparacion", "criterio"])
)

In [ ]:
sens_deltas = pd.read_csv(OUTPUTS["sens_delta_by_fold"])
criterion_colors = {"auc_max": "#0072B2", "ap_max": "#E69F00", "loss_min": "#CC79A7"}

fig, axes = plt.subplots(1, len(COMPARISONS), figsize=(11, 4.2), sharey=True)
for ax, comparison in zip(axes, COMPARISONS):
    for offset, criterion in zip((-0.2, 0, 0.2), CRITERION_LABELS):
        rows = sens_deltas[(sens_deltas["comparacion"] == comparison["id"]) & (sens_deltas["criterio"] == criterion)].sort_values("fold")
        ax.scatter(rows["fold"] + offset, rows["delta_ap"], color=criterion_colors[criterion], s=45, label=CRITERION_LABELS[criterion])
    ax.axhline(0, color="black", lw=1)
    ax.set_xticks(range(N_FOLDS))
    ax.set_xlabel("Fold")
    ax.set_title(comparison["id"], fontsize=10)
axes[0].set_ylabel("Δ AP (primera − segunda)")
axes[0].legend(fontsize=8)
fig.suptitle("Figura 3. Δ AP por fold según el criterio de checkpoint (descriptivo)", y=1.02)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "fig3_sensibilidad_delta_ap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Limitaciones

- E2 es estratificada y no por paciente (no hay identificadores de paciente): puede sobreestimar, y los cinco folds no son independientes entre sí porque comparten imágenes de entrenamiento.
- Hay una semilla por fold: no se separa la variabilidad por semilla de la variabilidad por fold. El entrenamiento no fuerza el modo determinista de cuDNN.
- Cada fold de validation tiene 21 o 22 positivas operacionales: un solo falso negativo cambia mucho el AP del fold. Con cinco folds, la regla de 4 de 5 es una convención del protocolo y no una prueba estadística.
- Las categorías 6 y 7 se agrupan en la clase positiva operacional; la categoría 6 no es malignidad histológica.
- Las 1000 imágenes de E2 incluyen las 200 de validation de E1, sobre las que se seleccionaron B2 y el criterio de checkpoint: E2 no es independiente de esa selección.
- La sensibilidad del criterio de checkpoint se calcula sobre los mismos folds en los que se elige la época; no es una evaluación independiente de cada criterio.
- Solo se usan archivos de validation; test no interviene.

## 9. Interpretación de resultados

*Pendiente.* Esta sección se redacta solo después de ejecutar el notebook, con las cifras de los archivos de `robustez_e2/analisis/` y sin modificar las reglas de lectura del protocolo.